# EP8 — Multiple Subplots for Manufacturing Monitoring

Lab นี้ใช้ข้อมูลสังเคราะห์จากโรงงานบรรจุภัณฑ์เพื่อฝึก `plt.subplots()`, Shared Axes, `GridSpec`, `subplot_mosaic()` และ `fig.add_axes()`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. สร้างข้อมูลหนึ่งกะการผลิต

In [ ]:
rng = np.random.default_rng(8)

minutes = np.arange(0, 480, 10)
shift_hour = minutes / 60

production_cycle = np.sin(
    2 * np.pi * (minutes + 30) / 480
)

line_a = (
    90
    + 7 * production_cycle
    + rng.normal(0, 2.0, minutes.size)
)
line_b = (
    85
    + 6 * np.sin(2 * np.pi * (minutes + 70) / 480)
    + rng.normal(0, 2.2, minutes.size)
)
line_c = (
    93
    + 5 * np.sin(2 * np.pi * (minutes + 10) / 480)
    + rng.normal(0, 1.8, minutes.size)
)

motor_temperature = (
    54
    + 0.032 * minutes
    + 1.8 * np.sin(2 * np.pi * minutes / 180)
    + rng.normal(0, 0.6, minutes.size)
)

vibration = np.clip(
    1.9
    + 0.0025 * minutes
    + 0.25 * np.sin(2 * np.pi * minutes / 120)
    + rng.normal(0, 0.09, minutes.size),
    0,
    None,
)

defect_rate = np.clip(
    1.8
    + 0.045 * (motor_temperature - 54)
    + 0.22 * np.sin(2 * np.pi * minutes / 160)
    + rng.normal(0, 0.12, minutes.size),
    0,
    None,
)

print("Samples:", minutes.size)

## 2. กริดพื้นฐาน 2×2

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 7),
    sharex=True,
    layout="constrained",
)

series = [
    (line_a, "Line A Output Rate", "Output rate (units/min)", "tab:blue"),
    (motor_temperature, "Motor Temperature", "Temperature (°C)", "tab:orange"),
    (vibration, "Motor Vibration", "Vibration (mm/s)", "tab:green"),
    (defect_rate, "Defect Rate", "Defect rate (%)", "tab:red"),
]

for ax, (values, title, ylabel, color) in zip(axes.flat, series):
    ax.plot(shift_hour, values, color=color, linewidth=2)
    ax.set(title=title, ylabel=ylabel)
    ax.grid(alpha=0.25)

for ax in axes[-1, :]:
    ax.set_xlabel("Shift hour")

fig.suptitle("Packaging Line A — Shift Monitoring")
plt.show()

## 3. Shared Axes สำหรับสามสายการผลิต

In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(10, 7),
    sharex=True,
    sharey=True,
    layout="constrained",
)

line_series = [
    (line_a, "Line A", "tab:blue"),
    (line_b, "Line B", "tab:orange"),
    (line_c, "Line C", "tab:green"),
]

for ax, (values, label, color) in zip(axes, line_series):
    ax.plot(shift_hour, values, color=color, linewidth=2)
    ax.axhline(90, color="gray", linestyle="--", linewidth=1.2)
    ax.set_ylabel(label)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Shift hour")
fig.suptitle("Output Rate by Production Line")
plt.show()

## 4. GridSpec สำหรับ Dashboard ที่แต่ละกราฟใช้พื้นที่ต่างกัน

In [ ]:
fig = plt.figure(figsize=(13, 7.5), layout="constrained")
grid = fig.add_gridspec(
    2,
    3,
    width_ratios=[1.3, 1.3, 1],
    height_ratios=[1.2, 1],
)

output_ax = fig.add_subplot(grid[0, :2])
summary_ax = fig.add_subplot(grid[0, 2])
temperature_ax = fig.add_subplot(grid[1, 0])
vibration_ax = fig.add_subplot(grid[1, 1])
quality_ax = fig.add_subplot(grid[1, 2])

for values, label in [(line_a, "Line A"), (line_b, "Line B"), (line_c, "Line C")]:
    output_ax.plot(shift_hour, values, label=label)
output_ax.set(title="Output Rate During the Shift", xlabel="Shift hour", ylabel="Units/min")
output_ax.legend(ncols=3)

summary_ax.barh(
    ["Line A", "Line B", "Line C"],
    [line_a.mean(), line_b.mean(), line_c.mean()],
    color=["tab:blue", "tab:orange", "tab:green"],
)
summary_ax.set(title="Average Output", xlabel="Units/min")

temperature_ax.plot(shift_hour, motor_temperature, color="tab:orange")
temperature_ax.set(title="Motor Temperature", xlabel="Shift hour", ylabel="°C")

vibration_ax.plot(shift_hour, vibration, color="tab:green")
vibration_ax.set(title="Vibration", xlabel="Shift hour", ylabel="mm/s")

quality_ax.plot(shift_hour, defect_rate, color="tab:red")
quality_ax.set(title="Defect Rate", xlabel="Shift hour", ylabel="%")

for ax in fig.axes:
    ax.grid(alpha=0.22)

fig.suptitle("Packaging Factory — Shift Overview")
plt.show()

## 5. Subplot Mosaic แบบตั้งชื่อพื้นที่

In [ ]:
layout = [
    ["output", "output", "quality"],
    ["temperature", "vibration", "quality"],
]

fig, axes = plt.subplot_mosaic(
    layout,
    figsize=(13, 7),
    width_ratios=[1.2, 1.2, 1],
    layout="constrained",
)

for values, label in [(line_a, "Line A"), (line_b, "Line B"), (line_c, "Line C")]:
    axes["output"].plot(shift_hour, values, label=label)
axes["output"].set(title="Output Rate", xlabel="Shift hour", ylabel="Units/min")
axes["output"].legend(ncols=3)

axes["temperature"].plot(shift_hour, motor_temperature, color="tab:orange")
axes["temperature"].set(title="Motor Temperature", xlabel="Shift hour", ylabel="°C")

axes["vibration"].plot(shift_hour, vibration, color="tab:green")
axes["vibration"].set(title="Vibration", xlabel="Shift hour", ylabel="mm/s")

axes["quality"].plot(defect_rate, shift_hour, color="tab:red")
axes["quality"].set(title="Quality Trend", xlabel="Defect rate (%)", ylabel="Shift hour")

for ax in axes.values():
    ax.grid(alpha=0.22)

fig.suptitle("Packaging Line Monitoring Mosaic")
plt.show()

## 6. Manual Axes และ Inset

In [ ]:
fig = plt.figure(figsize=(11, 6))
main_ax = fig.add_axes([0.09, 0.13, 0.82, 0.77])
inset_ax = fig.add_axes([0.61, 0.57, 0.25, 0.24])

main_ax.plot(shift_hour, line_a, color="tab:blue", linewidth=2.5)
main_ax.axhline(90, color="gray", linestyle="--")
main_ax.set(title="Line A Output Rate", xlabel="Shift hour", ylabel="Output rate (units/min)")
main_ax.grid(alpha=0.25)

inset_ax.plot(shift_hour, defect_rate, color="tab:red")
inset_ax.set_title("Defect Rate", fontsize=10)
inset_ax.tick_params(labelsize=8)
inset_ax.grid(alpha=0.2)

plt.show()

## แบบทดลอง

1. เปลี่ยนกริด 2×2 เป็น 1×4
2. ทดลอง `width_ratios=[3, 1]`
3. เพิ่ม Missing Data ใน Vibration ด้วย `np.nan`
4. สร้าง Mosaic ใหม่ที่กราฟ Output กินแถวบนทั้งหมด